# Applied Data Science Capstone — 1. Data Collection (SpaceX API & Web Scraping)

## Objective
Collect Falcon 9 launch records from two sources, as in the IBM capstone labs:

1. **SpaceX REST API v4** (`api.spacexdata.com/v4`) — structured launch, rocket, launchpad, payload and core records.
2. **Wikipedia** — the *List of Falcon 9 and Falcon Heavy launches* page, scraped for the landing-outcome column that the API does not expose directly.

The curated result of this pipeline is `data/dataset_spacex.csv`: **90 Falcon 9 launches** (Falcon 1 flights excluded).

## 1.1 SpaceX API v4 — request pattern
The v4 API exposes `/launches/query` (POST with a Mongo-style query body).
We request Falcon 9 launches and join the referenced `rocket`, `launchpad`,
`payloads` and `cores` documents. The cell below tries the live endpoint and
falls back to a documented sample record when the network is unavailable.

In [1]:
import json
import requests
import pandas as pd

API = "https://api.spacexdata.com/v4"

def fetch_launch_sample():
    """Return one Falcon 9 launch document (live call, documented fallback)."""
    try:
        r = requests.post(
            f"{API}/launches/query",
            json={"query": {"name": {"$regex": "Falcon 9"}},
                  "options": {"limit": 1, "populate": ["rocket", "launchpad"]}},
            timeout=15,
        )
        r.raise_for_status()
        print("live API call OK")
        return r.json()["docs"][0]
    except Exception as e:
        print(f"live API unreachable ({type(e).__name__}); using documented sample record")
        # Documented shape of a v4 /launches document (fields kept by the pipeline)
        return {
            "name": "Falcon 9 Block 5 | Starlink Group 4-23",
            "date_utc": "2022-08-28T03:41:00.000Z",
            "rocket": {"name": "Falcon 9", "type": "rocket"},
            "launchpad": {"name": "CCSFS SLC 40", "locality": "Cape Canaveral",
                          "latitude": 28.5632, "longitude": -80.5768},
            "payloads": [{"name": "Starlink Group 4-23", "mass_kg": 14800, "orbit": "VLEO"}],
            "cores": [{"core": "B1069", "flight": 1, "gridfins": True,
                       "legs": True, "reused": False,
                       "landing_attempt": True, "landing_success": True,
                       "landing_type": "ASDS"}],
            "success": True,
        }

sample = fetch_launch_sample()
print(json.dumps(sample, indent=2)[:1200])

live API unreachable (ReadTimeout); using documented sample record
{
  "name": "Falcon 9 Block 5 | Starlink Group 4-23",
  "date_utc": "2022-08-28T03:41:00.000Z",
  "rocket": {
    "name": "Falcon 9",
    "type": "rocket"
  },
  "launchpad": {
    "name": "CCSFS SLC 40",
    "locality": "Cape Canaveral",
    "latitude": 28.5632,
    "longitude": -80.5768
  },
  "payloads": [
    {
      "name": "Starlink Group 4-23",
      "mass_kg": 14800,
      "orbit": "VLEO"
    }
  ],
  "cores": [
    {
      "core": "B1069",
      "flight": 1,
      "gridfins": true,
      "legs": true,
      "reused": false,
      "landing_attempt": true,
      "landing_success": true,
      "landing_type": "ASDS"
    }
  ],
  "success": true
}


## 1.2 Normalising the API documents
Each launch document nests rocket / launchpad / payload / core objects.
The pipeline flattens one row per launch with the columns the later labs need:
`FlightNumber, Date, BoosterVersion, PayloadMass, Orbit, LaunchSite, Flights,
GridFins, Reused, Legs, LandingPad, Block, ReusedCount, Serial, Customer,
Latitude, Longitude`.

In [2]:
def normalise(doc):
    core = (doc.get("cores") or [{}])[0]
    payload = (doc.get("payloads") or [{}])[0]
    pad = doc.get("launchpad", {}) if isinstance(doc.get("launchpad"), dict) else {}
    return {
        "FlightNumber": None,  # assigned sequentially after filtering Falcon 9
        "Date": doc.get("date_utc", "")[:10],
        "BoosterVersion": doc.get("name", ""),
        "PayloadMass": payload.get("mass_kg"),
        "Orbit": payload.get("orbit"),
        "LaunchSite": pad.get("name"),
        "Flights": core.get("flight"),
        "GridFins": core.get("gridfins"),
        "Reused": core.get("reused"),
        "Legs": core.get("legs"),
        "LandingPad": core.get("landpad"),
        "Serial": core.get("core"),
        "LandingSuccess": core.get("landing_success"),
        "LandingType": core.get("landing_type"),
        "MissionSuccess": doc.get("success"),
        "Latitude": pad.get("latitude"),
        "Longitude": pad.get("longitude"),
    }

print(normalise(sample))

{'FlightNumber': None, 'Date': '2022-08-28', 'BoosterVersion': 'Falcon 9 Block 5 | Starlink Group 4-23', 'PayloadMass': 14800, 'Orbit': 'VLEO', 'LaunchSite': 'CCSFS SLC 40', 'Flights': 1, 'GridFins': True, 'Reused': False, 'Legs': True, 'LandingPad': None, 'Serial': 'B1069', 'LandingSuccess': True, 'LandingType': 'ASDS', 'MissionSuccess': True, 'Latitude': 28.5632, 'Longitude': -80.5768}


## 1.3 Wikipedia webscraping — landing outcomes
The public API does not expose a clean *landing outcome* column, so the course
scrapes the Wikipedia table *List of Falcon 9 and Falcon Heavy launches*
(`requests` + `BeautifulSoup`, parsing the wikitable rows into
*Flight No., Date, Booster version, Payload, Orbit, Customer, Launch outcome,
Booster landing*). Historical note used in the report: SpaceX's **first static
fire test was conducted in 2006**.

The scraper pattern (run where network access is available):

In [3]:
scraper_pattern = """
import requests
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
html = requests.get(URL, timeout=30).text
soup = BeautifulSoup(html, "html.parser")
tables = soup.find_all("table", {"class": "wikitable"})
print("wikitable count:", len(tables))
# Each launch table row -> [flight_no, date, booster_version, payload,
#                           orbit, customer, launch_outcome, booster_landing]
"""
print(scraper_pattern)
print("BeautifulSoup available:", end=" ")
try:
    import bs4
    print("yes", bs4.__version__)
except ImportError:
    print("no (pattern documented for an environment with network access)")


import requests
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
html = requests.get(URL, timeout=30).text
soup = BeautifulSoup(html, "html.parser")
tables = soup.find_all("table", {"class": "wikitable"})
print("wikitable count:", len(tables))
# Each launch table row -> [flight_no, date, booster_version, payload,
#                           orbit, customer, launch_outcome, booster_landing]

BeautifulSoup available: no (pattern documented for an environment with network access)


## 1.4 Curated dataset
Merging the API records with the Wikipedia landing-outcome column and keeping
**only Falcon 9 flights** (Falcon 1 excluded) yields the analysis dataset:
**90 launches**. It is stored as `data/dataset_spacex.csv` and is the input of
every later notebook.

In [4]:
df = pd.read_csv("../data/dataset_spacex.csv")
print("shape:", df.shape)
print("columns:", list(df.columns))
print()
print(df.head(3).to_string())
print()
print("Falcon 9 launches:", (df["BoosterVersion"].str.contains("Falcon 9")).sum())
print("Date range:", df["Date"].min(), "->", df["Date"].max())

shape: (90, 23)
columns: ['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'LandingType', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'RecoveryShip', 'Block', 'ReusedCount', 'Serial', 'Customer', 'StaticFire', 'MissionOutcome', 'Latitude', 'Longitude', 'Class', 'Outcome']

   FlightNumber        Date        BoosterVersion  PayloadMass Orbit   LaunchSite LandingType  Flights  GridFins  Reused  Legs LandingPad RecoveryShip  Block  ReusedCount Serial Customer  StaticFire MissionOutcome  Latitude  Longitude  Class     Outcome
0             1  2019-11-08      Falcon 9 Block 5       4115.3  VLEO  CCAFS LC-40        ASDS        4     False    True  True        NaN       OCISLY      5            0  B1066      SES        True        Success   28.5623   -80.5774      1   True ASDS
1             2  2017-06-06      Falcon 9 Block 5       1229.5   SSO  CCAFS LC-40       Ocean        5      True   False  True        NaN          NaN      5            0  B106